# Notebook 3: Model Training & Evaluation Pipeline

## Objective
In this notebook, we will train and compare various machine learning models to predict customer churn. 
We will evaluate both baseline and advanced ensemble models using the SMOTE balanced dataset.

**Models to evaluate:**
1. **Logistic Regression:** The interpretable baseline.
2. **Random Forest:** A robust bagging ensemble model.
3. **XGBoost:** A highly performant boosting ensemble model.
4. **LightGBM:** A fast, leaf-wise boosting ensemble model optimized for tabular data.

In [2]:
# Install advanced boosting libraries automatically if you are missing
!pip install xgboost lightgbm -q

In [5]:
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') 

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load the processed datasets from the file
with open('../data/processed/processed_data.pkl', 'rb') as f:
    X_train_balanced, y_train_balanced, X_test_scaled, y_test = pickle.load(f)

print("Downloaded processed data successfully")
print(f"Shape of Training Data: {X_train_balanced.shape}")
print(f"Shape of Test Data:  {X_test_scaled.shape}")

Downloaded processed data successfully
Shape of Training Data: (13598, 21)
Shape of Test Data:  (2026, 21)


In [8]:
# Define the models in a dictionary
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

# Initialize an empty list to store the evaluation metrics
results = []

print(" Starting Model Training and Evaluation Pipeline...\n")

# Loop through each model, train, predict, and evaluate
for name, model in models.items():
    print(f" Training {name}...")
    
    # Train the model using the SMOTE balanced data (from Notebook 2)
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predict on the unseen scaled test data
    y_pred = model.predict(X_test_scaled)
    
# Calculate metrics focusing on Churners (now encoded as 1)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1) # Sửa thành 1
    rec = recall_score(y_test, y_pred, pos_label=1)   # Sửa thành 1
    f1 = f1_score(y_test, y_pred, pos_label=1)       # Sửa thành 1
    
    # Save the metrics
    results.append({
        "Model Name": name,
        "Accuracy": acc,
        "Precision (Churn)": prec,
        "Recall (Churn)": rec,
        "F1-Score (Churn)": f1
    })

print("\n All models trained and evaluated successfully.")

 Starting Model Training and Evaluation Pipeline...

 Training Logistic Regression...
 Training Random Forest...
 Training XGBoost...
 Training LightGBM...

 All models trained and evaluated successfully.


## Model Comparison & Leaderboard
Convert results into a DataFrame and sort them by **F1-Score** to identify the best performing model for predicting customer churn.

In [10]:
# Convert the list of dictionaries into a Pandas DataFrame
results_df = pd.DataFrame(results)

# Sort by F1-Score (descending) to see the best model at the top
results_df = results_df.sort_values(by="F1-Score (Churn)", ascending=False).reset_index(drop=True)

# Highlight the highest values in each column for easy reading
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]

print("MODEL LEADERBOARD (Focusing on Attrited Customers - Class 0):")
display(results_df.style.apply(highlight_max, subset=['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-Score (Churn)']))

MODEL LEADERBOARD (Focusing on Attrited Customers - Class 0):


,Model Name,Accuracy,Precision (Churn),Recall (Churn),F1-Score (Churn)
0,LightGBM,0.968411,0.911672,0.889231,0.900312
1,XGBoost,0.967423,0.919094,0.873846,0.895899
2,Random Forest,0.956565,0.864615,0.864615,0.864615
3,Logistic Regression,0.873643,0.574514,0.818462,0.675127
